In [ ]:
# Imports
import numpy as np
from os import getcwd, makedirs
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.optimize import curve_fit

# Enable local imports
from os.path import abspath, join as path_join
from sys import path as syspath
module_path = abspath(path_join('..'))
if module_path not in syspath:
    syspath.append(module_path)
from utilities.data_handling.csv_reader import read_graph_data
from utilities.time.time_strings import get_string_date

In [ ]:
# "Config" parameters
cwd = Path(getcwd())
base_working_path = Path(cwd / "temp")
todays_working_dir = base_working_path / get_string_date()
data_dir = base_working_path / "data"
higgs_data_dir = data_dir / "higgs"
x_data_csv_path = higgs_data_dir / "higgs_data_X.csv"
processed_x_npy_path = todays_working_dir / "X_proc.npy"
data_dir = base_working_path / "data"

In [ ]:
## Process the originally extracted data point from the graph of the Higgs discovery
x_data, y_data = read_graph_data(x_data_csv_path)

# Convert to numpy array
x_meas_data = np.array([x_data], dtype=float)

# Round to third digit after 3rd decimal point
x_meas_data = np.around(x_meas_data, decimals=3)

# Save X for prediction
makedirs(todays_working_dir, exist_ok=True)
np.save(processed_x_npy_path, x_meas_data)

Here you should run some net over the processed X through `DDP.py` and return with resulting B and Z files.

In [ ]:
# Additional "Config" parameters
prediction_z_output_path = higgs_data_dir / "prediction_higgs_over_function_with_signal" / "pred_z.npy"
prediction_b_output_path = higgs_data_dir / "prediction_higgs_over_function_with_signal" / "pred_b.npy"
figure_output_path = higgs_data_dir / "ZNet3_DMfunc_mix_width_sig.png"

In [ ]:
# Styling parameters
fontsize = 18
linestyle = 'dotted'
data_color = 'black'
prediction_color = 'red'
fit_color = 'blue'

In [ ]:
# Additional constants
mass_GeV = np.array([101.0199557,103.0155211,105.0110865,107.0066519,109.0022173,110.9977827,113.0266075,114.9889135,116.9844789,119.0133038,121.0088692,123.0044346,125,126.9955654,129.0243902,130.9866962,133.0155211,135.0110865,137.0066519,139.0022173,140.9977827,143.0266075,144.9889135,147.0177384,149.0133038,151.0088692,153.0044346,155,157.0288248,158.9911308])
min_x, max_x = 100, 155
smoothed_mass_GeV = np.linspace(min_x, max_x, 100)

In [ ]:
# Load data and estimations
x_meas_data = np.load(processed_x_npy_path, allow_pickle=True)[0]
z_pred_significance = np.load(prediction_z_output_path, allow_pickle=True)[0]
b_pred_background = np.load(prediction_b_output_path, allow_pickle=True)[0]
s_pred_signal = x_meas_data - b_pred_background

In [ ]:
# Argmax
max_pred_significance = np.argmax(z_pred_significance)

# Define the Gaussian function
def gauss(x, a, b, c):
    y = a*np.exp(-1*(x-c)**2/(2*b)**2)# + D
    return y

# Fit a Gaussian to the predicted signal
gaussian_fit_parameters, covariance = curve_fit(gauss, mass_GeV, s_pred_signal, p0=[122, 2, 128])
# gauss_fit_A, fit_B, fit_C = gaussian_fit_parameters
gaussian_fit_y = gauss(smoothed_mass_GeV, *gaussian_fit_parameters)

# Fit a 4th degree polynomial to the smoothly falling background, as they
polyfit_parameters = np.polyfit(mass_GeV, b_pred_background, 4)
fitted_polynomial = np.poly1d(polyfit_parameters)
polynomial_fit_y = fitted_polynomial(smoothed_mass_GeV)

# Plot structure
fig = plt.figure(figsize=(10,9), dpi=120)
gs = GridSpec(3, 1, hspace=0, height_ratios=[3,2,2])
graphs = [fig.add_subplot(gs[i]) for i in range(3)]
pred_b_graph, pred_s_graph, pred_z_graph = graphs

# Predicted B graph
pred_b_graph.scatter(mass_GeV, b_pred_background, label='predicted background', c=prediction_color)
pred_b_graph.scatter(mass_GeV, x_meas_data, label='experimental data', c=data_color)
pred_b_graph.plot(smoothed_mass_GeV, polynomial_fit_y, label="4th degree polynomial fitted to the experimental data", c=fit_color)
pred_b_graph.set_ylabel('Events/2GeV',fontsize=fontsize)
pred_b_graph.tick_params(axis='y', which='major', labelsize=fontsize)
pred_b_graph.set_xticklabels([])
pred_b_graph.axvline(max_pred_significance ,linestyle=linestyle)

# Pred S graph
pred_s_graph.scatter(mass_GeV, s_pred_signal, c=prediction_color, label="predicted signal")
pred_s_graph.plot(smoothed_mass_GeV, gaussian_fit_y, '-', label='gaussian fit of the predicted signal', c=fit_color)
pred_s_graph.set_ylabel('Events/2GeV',fontsize=fontsize)
pred_s_graph.tick_params(axis='y', which='major', labelsize=fontsize)
pred_s_graph.set_xticklabels([])

# Predicted Z significance graph
pred_z_graph.plot(mass_GeV, z_pred_significance, c=prediction_color, label='predicted significance')
pred_z_graph.set_ylabel('Significance',fontsize=fontsize)
pred_z_graph.set_xlabel('GeV',fontsize=fontsize)
pred_z_graph.set_ylim(-1.5,10)
pred_z_graph.tick_params(axis='both', which='major', labelsize=fontsize)
fig.savefig(figure_output_path,facecolor='white', transparent=False)

for graph in graphs:
    graph.set_xlim(min_x, max_x)
    graph.axvline(mass_GeV[max_pred_significance],linestyle = linestyle)
    graph.grid(True)
    graph.legend()


In [ ]:
#save figure in a path

path = "/storage/agrp/amitshk/DDP/plots/net_systematics_v10/Higgs/1_sig"
plt.savefig(path + "/Znet3_DMfunc_1_width_sig.png")